# 언어 모델

**언어 모델(Language Model)** 은 앞에 주어진 토큰 문맥으로 다음 토큰의 조건부확률을 계산하는 모델이다. `나는 자연어 처리를`이라는 문맥에서 `배운다`, `공부한다` 같은 후보마다 확률을 만들며, 학습 데이터에서 관찰한 텍스트의 순서 패턴을 이 확률에 반영한다.

## 다음 토큰 예측과 생성 흐름

다음 토큰 예측을 한 번 수행하면 자동완성이 되고, 선택한 토큰을 다시 문맥에 넣어 반복하면 가변 길이 문장을 생성할 수 있다.

- **문맥 입력**: 지금까지의 토큰 ID를 모델에 전달한다.
- **로짓 계산**: 마지막 입력 위치에서 어휘의 모든 토큰 후보에 대한 정규화 전 점수를 만든다.
- **확률 변환**: softmax로 로짓을 합이 1인 다음 토큰 조건부확률로 바꾼다.
- **토큰 선택과 반복**: greedy 또는 sampling으로 ID 하나를 고르고 문맥 뒤에 붙여 다음 예측을 이어 간다.

## 대표 사용처

다음 토큰 조건부확률은 새로운 텍스트를 순서대로 만들어야 하는 작업의 공통 기반이다.

- 검색창·문서 작성기의 다음 단어 자동완성
- GPT 계열의 문장 생성과 대화 응답
- 기계 번역·요약 모델 Decoder의 출력 토큰 생성

## 장점과 한계

언어 모델은 하나의 확률 분포로 여러 후보를 비교하고 출력 길이를 미리 정하지 않아도 된다는 장점이 있다. 반면 학습 데이터의 편향을 반영할 수 있고, 자동회귀 생성 중 앞선 오류가 뒤 문맥에 누적되며, 높은 다음 토큰 확률이 문장의 사실성까지 보장하지는 않는다.

## 이번 노트북에서 확인할 내용

GPT-2를 직접 학습하지 않고 사전학습된 소형 checkpoint를 불러와 추론 흐름에 집중한다.

- 문자열이 토큰 ID와 `attention_mask`로 바뀌는 과정
- 마지막 위치 로짓에서 top-k 토큰·ID·확률을 연결하는 과정
- greedy와 sampling이 같은 확률을 서로 다르게 사용하는 방식

BERT는 가린 토큰을 양쪽 문맥으로 복원한다는 점에서 여기서 다루는 왼쪽 문맥 기반 생성과 예측 방향이 다르다. 먼저 다음 절에서 문장 확률을 위치별 다음 토큰 확률로 나누는 이유를 살펴보고, 빈도와 신경망 가중치를 직접 학습하는 과정은 뒤의 N-gram과 NNLM 노트북에서 이어서 다룬다.


## 01. 왜 다음 토큰을 예측하는가

문장 전체를 하나의 경우로 외우면 처음 보는 문장에 확률을 줄 수 없다. 언어 모델은 긴 문장의 확률을 각 위치에서 **지금까지의 문맥을 보았을 때 다음 토큰이 나올 조건부확률**로 나눈다. 이 분해 덕분에 같은 다음 토큰 예측 문제를 모든 토큰 위치에 반복하여 학습할 수 있다.

토큰 시퀀스를 $w_1, w_2, \ldots, w_T$라고 하면 문장 확률은 연쇄법칙으로 다음과 같이 표현된다.

$$
P(w_1, w_2, \ldots, w_T)
= \prod_{t=1}^{T} P(w_t \mid w_1, \ldots, w_{t-1})
$$

예를 들어 `<BOS> 나는 자연어 처리를 배운다 <EOS>`의 확률은 `나는`의 확률, 앞의 `나는`을 보았을 때 `자연어`의 확률, 앞의 모든 토큰을 보았을 때 다음 토큰의 확률을 차례로 곱한 값이다. `<EOS>`까지 포함해야 문장이 어디에서 끝나는지도 확률에 반영된다.

각 위치의 조건부확률이 예시로 `0.6`, `0.5`, `0.4`라면 세 선택이 모두 이어질 확률은 `0.6 × 0.5 × 0.4 = 0.12`이다. 실제 언어 모델은 어휘 전체에 확률을 나누어 주며, 긴 문장은 곱하는 항이 많아 확률이 작아진다. 따라서 길이가 다른 문장을 단순 문장 확률만으로 비교하면 안 되며, 이 문제는 뒤의 N-gram 단원에서 perplexity와 함께 다룬다.

다음 절에서는 이 위치별 예측이 학습 때는 정답과 손실을 만드는 데 쓰이고, 추론 때는 선택한 토큰을 다음 입력으로 만드는 데 쓰인다는 차이를 연결한다.


## 02. 학습과 추론은 무엇이 다른가

**학습(Training)** 에서는 말뭉치의 실제 다음 토큰과 모델의 예측을 비교해 손실을 계산한다. 예측이 틀릴수록 손실이 커지고, 역전파와 optimizer가 가중치를 수정한다. 하나의 토큰 시퀀스를 한 칸 어긋나게 놓으면 각 입력 위치가 바로 다음 토큰의 정답을 갖는다.

```text
입력 위치 : <BOS> | 나는   | 자연어 | 처리를
정답 위치 : 나는  | 자연어 | 처리를 | 배운다
학습 결과 : 다음 정답의 확률을 높이도록 가중치를 갱신한다.
```

**추론(Inference)** 에서는 정답 다음 토큰이 없다. 모델이 만든 확률 분포에서 토큰 하나를 선택하고, 그 토큰을 문맥 뒤에 붙여 다음 예측의 입력으로 사용한다. 가중치는 갱신하지 않으며, 앞에서 잘못 선택한 토큰도 다음 문맥에 포함되므로 생성이 진행될수록 오류가 이어질 수 있다.

이 노트북의 `from_pretrained()`는 저장된 tokenizer 설정과 학습된 가중치를 내려받아 읽는 과정이다. `model.eval()`과 `torch.no_grad()`를 사용한 뒤의 계산은 이미 학습된 모델의 **추론**이다. 따라서 다운로드가 완료되거나 자연스러운 문장이 출력되었다는 사실을 현재 노트북에서 새로 학습한 결과로 해석하면 안 된다.

학습과 추론은 가중치 갱신 여부가 다르지만 두 단계 모두 마지막에는 어휘 전체의 로짓을 만든다. 다음 절에서는 모델 구조가 달라도 공통으로 나타나는 이 출력 형식을 정리한다.


## 03. 서로 다른 언어 모델이 만드는 공통 결과

N-gram은 말뭉치의 등장 횟수로 조건부확률을 추정하고, NNLM은 고정 길이 문맥의 임베딩을 신경망에 전달하며, GPT-2는 앞에 나온 여러 토큰을 Transformer로 처리한다. 내부 계산은 다르지만 마지막에는 현재 어휘의 각 토큰에 대한 **로짓(Logit)** 과 **다음 토큰 확률**을 만든다.

```text
문자열 문맥
   ↓ tokenizer
토큰 ID (B, T)
   ↓ N-gram / NNLM / Transformer
어휘 전체 로짓 (B, V)
   ↓ softmax
P(next_token | context) (B, V)
   ↓ greedy 또는 sampling
선택한 토큰 ID → 문맥 뒤에 추가 → 다음 위치 예측
```

shape의 각 축은 뒤의 GPT-2 출력과 N-gram·NNLM 결과를 같은 언어로 비교하기 위한 기준이다.

- `B`: 한 번에 처리하는 문장 수인 배치 축이다.
- `T`: 문장 안의 입력 토큰 위치 축이다.
- `V`: 다음 토큰 후보가 놓인 어휘 축이다.

이 노트북의 핵심은 특정 모델 구조를 외우는 것이 아니라 `문맥 → 어휘별 점수 → 확률 → 토큰 선택 → 다음 문맥`이라는 공통 흐름을 추적하는 것이다. 이제 이 흐름을 실제 사전학습 GPT-2의 tokenizer와 모델 출력으로 확인한다.


## 04. 사전학습 GPT-2 준비

### GPT-2란 무엇인가

**GPT-2(Generative Pre-trained Transformer 2)** 는 앞에 나온 토큰만 보고 다음 토큰을 예측하도록 대규모 영어 텍스트에서 미리 학습한 생성 언어 모델이다. 예측한 토큰을 문맥 뒤에 붙이고 같은 과정을 반복하여 문장을 생성한다.

- **Generative**: 다음 토큰을 반복해서 선택하여 새로운 텍스트를 생성한다.
- **Pre-trained**: 현재 노트북에서 처음부터 학습하지 않고, 대규모 문서로 미리 학습된 가중치를 사용한다.
- **Transformer**: 여러 앞 토큰의 관계를 처리하는 신경망 구조이다. 내부 계산은 이후 Transformer 단원에서 다룬다.

GPT-2의 학습 방식을 **Causal Language Modeling**이라고 한다. 각 위치에서 오른쪽의 미래 정답은 보지 못하게 가리고 왼쪽에 이미 나온 토큰만 이용해 바로 다음 토큰을 예측한다.

```text
입력 문맥: Natural language processing helps computers
                                              ↓
GPT-2 출력: 다음에 올 수 있는 어휘 전체의 점수와 확률
                                              ↓
토큰 선택: learn / understand / to / ...
```

### 현재 코드에서 불러오는 것

`openai-community/gpt2`는 약 1억 2천만 개의 파라미터를 가진 GPT-2의 모델 설정·어휘·학습 가중치를 저장한 checkpoint이다. 최초 실행에는 파일 다운로드가 필요하며 같은 환경에서는 cache의 파일을 다시 사용한다.

같은 checkpoint 이름으로 tokenizer와 모델을 준비해야 입력 ID와 출력 로짓의 어휘 위치가 정확히 대응한다.

- `AutoTokenizer`: 문자열을 GPT-2 어휘의 토큰 ID와 실제 토큰 위치를 표시하는 `attention_mask`로 변환한다.
- `AutoModelForCausalLM`: 앞 토큰만 사용해 다음 토큰 로짓을 계산하는 GPT-2 구조와 사전학습 가중치를 불러온다.
- `model.eval()`과 `torch.no_grad()`: dropout과 기울기 기록을 학습이 아닌 추론 방식으로 제어한다.
- **EOS(End Of Sequence)**: 생성의 끝을 나타내는 특수 토큰이며 GPT-2에서는 `<|endoftext|>`를 사용한다.

이어지는 코드에서는 checkpoint·어휘 크기·EOS ID를 출력한다. 이 값들이 올바르게 로드되어야 뒤에서 로짓의 어휘 열을 tokenizer의 토큰 ID로 복원할 수 있다.


In [2]:
%pip install transformers

  Using cached transformers-5.14.1-py3-none-any.whl.metadata (32 kB)
  Using cached regex-2026.7.19-cp312-cp312-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.27.0-py3-none-any.whl.metadata (15 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached hf_xet-1.5.2-cp38-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached transformers-5.14.1-py3-none-any.whl (11.6 MB)
   ---------------------------------------- 0.0/774.9 kB ? eta -:--:--
   --------------------------- ------------ 524.3/774.9 kB 8.5 MB/s eta 0:00:01
   --------------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

model.eval()

pad_token_id = tokenizer.eos_token_id

print("checkpoint:", MODEL_NAME)
print("어휘 크기:", tokenizer.vocab_size)
print("EOS 토큰과 ID:", repr(tokenizer.eos_token), tokenizer.eos_token_id)


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

C:\Users\playdata2\miniforge3\envs\pystudy_env\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\playdata2\.cache\huggingface\hub\models--openai-community--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

checkpoint: openai-community/gpt2
어휘 크기: 50257
EOS 토큰과 ID: '<|endoftext|>' 50256


## 05. 다음 토큰 top-k 확률

GPT-2는 입력의 각 토큰 위치마다 어휘 전체의 로짓을 출력한다. 문장 바로 다음 토큰을 예측하려면 마지막 입력 위치의 로짓만 선택하고, `softmax`를 어휘 축에 적용해 합이 `1`인 조건부확률 분포로 바꾼다.

```text
입력 토큰 ID       : (배치 B, 입력 위치 T)
모델 전체 로짓     : (배치 B, 입력 위치 T, 어휘 V)
마지막 위치 로짓   : (배치 B, 어휘 V)
다음 토큰 확률     : (배치 B, 어휘 V)
top-k 값과 ID      : (배치 B, 후보 k)
```

`torch.topk()`는 같은 순위에 놓인 두 텐서를 반환한다.

- `top_probabilities`: 선택된 후보의 확률값이다.
- `top_token_ids`: 그 확률이 놓였던 어휘 열의 ID이다.

top-1 하나만 보면 최종 선택만 알 수 있지만, 여러 후보의 ID를 문자열로 복원하고 확률을 함께 읽으면 모델이 다음 표현 사이에서 확률을 어떻게 나누었는지 확인할 수 있다. 출력에서는 전체 로짓의 `(B, T, V)`와 마지막 위치 후보의 실제 토큰 조각을 함께 대조한다.


In [5]:
# 마지막 입력 위치의 어휘 로짓을 확률로 바꾸고 상위 후보를 복원한다.
prompt = "Natural language processing helps computers"

# tokenizer == gpt-2 전용 토크나이저(학습된 사전이 포함되어 있음)

# return_tensors="pt" -> 토큰 ID 목록에 배치축(배치 크기)를 포함한 텐서 반환
inputs = tokenizer(prompt, return_tensors="pt") # 문장 1개 토큰 5

# 가중치 갱신에 필요한 기울기 저장 X -> 메모리를 아낌
with torch.no_grad():
    # 사전 학습 모델 gpt-2 생성
    # + 토큰화된 프롬프트의 토큰 ID 목록도 전달
    model_output = model(**inputs)
    # 결과 : logits 반환
    # -> logits에는 각 입력 위치가 다음 토큰을 예측한 정규화 점수가 저장된다.

# (배치 B, 입력 위치 T, 어휘 V)
# 마지막 입력 토큰인 'computers' 다음에 등장할 단어를
# gpt-2가 학습한 어휘 사전(50257)에 모든 단어에 대하여 logit 점수를 반환
last_token_logits = model_output.logits[:, -1, :]

# logits를 확률로 변경 -> 50257개의 확률을 모두 더하면 1
next_token_probabilities = torch.softmax(last_token_logits, dim=-1)

# 확률이 가장 높은 5개의 확률과 index 번호를 반환
top_probabilities, top_token_ids = torch.topk(
    next_token_probabilities,
    k=5,
    dim=-1,
)

input_token_ids = inputs["input_ids"][0].tolist()
input_tokens = [tokenizer.decode([token_id]) for token_id in input_token_ids]

print("입력 문장:", prompt)
print("입력 토큰:", input_tokens)
print("입력 토큰 ID:", input_token_ids)
print("입력 ID shape:", tuple(inputs["input_ids"].shape))
print("전체 로짓 shape:", tuple(model_output.logits.shape))
print("다음 토큰 후보:")

for rank, (token_id, probability) in enumerate(
    zip(top_token_ids[0].tolist(), top_probabilities[0].tolist()),
    start=1,
):
    token_text = tokenizer.decode([token_id])
    print(f"  {rank}. {token_text!r:<16} ID={token_id:<6} 확률={probability:.4f}")


입력 문장: Natural language processing helps computers
입력 토큰: ['Natural', ' language', ' processing', ' helps', ' computers']
입력 토큰 ID: [35364, 3303, 7587, 5419, 9061]
입력 ID shape: (1, 5)
전체 로짓 shape: (1, 5, 50257)
다음 토큰 후보:
  1. ' learn'         ID=2193   확률=0.1877
  2. ' understand'    ID=1833   확률=0.1136
  3. ' to'            ID=284    확률=0.1119
  4. ' do'            ID=466    확률=0.0312
  5. ' recognize'     ID=7564   확률=0.0294


### 06. Greedy 생성과 Sampling 생성

**Greedy decoding** 은 매 위치에서 확률이 가장 높은 토큰 하나를 선택한다. 같은 모델과 입력에서는 결과가 일정하지만, 한 번 선택한 top-1이 다음 문맥을 결정하므로 반복적이거나 단조로운 문장으로 이어질 수 있다.

**Sampling** 은 확률 분포에서 다음 토큰을 뽑는다. 후보 분포를 그대로 사용하기보다 다음 설정으로 선택 범위를 조절할 수 있다.

- `temperature`: 낮을수록 높은 로짓에 더 집중하고, 높을수록 낮은 로짓 후보도 선택될 가능성이 커진다.
- `top_k`: 확률 상위 `k`개 토큰만 sampling 후보로 남긴다.
- `top_p`: 확률이 높은 순서대로 더한 누적확률이 `p`에 도달하는 작은 후보 집합을 사용한다.

greedy는 일관성이 중요할 때 출발점으로 쓰기 쉽고, sampling은 여러 표현을 만들고 싶을 때 사용한다. 이어지는 코드는 같은 입력·모델·생성 길이를 유지하고 선택 규칙만 바꾸므로 두 결과의 차이를 선택 단계의 효과로 해석할 수 있다. 다만 두 방법 모두 모델의 학습 데이터 패턴을 이용할 뿐 사실성·안전성·적절성을 보장하지 않는다.


In [7]:
generation_options = {
    "max_new_tokens": 24, # 생성할 토큰 최대 수
    "pad_token_id": pad_token_id, # 0
    "repetition_penalty": 1.1, # 반복 패널티 (기본값1, 1 초과시 패널티 부여)
}

# Greedy 방식 생성
with torch.no_grad():
    greedy_ids = model.generate(
        **inputs, # tokenizer 결과 (토큰 ID 목록)
        do_sample=False, # Greedy 방식 선택 (True는 sampling)
        **generation_options, # 다음 토큰 생성 옵션 (최대 토큰 수)
    )

# Sampling 방식
GENERATION_SEED = 42
torch.manual_seed(GENERATION_SEED)

with torch.no_grad():
    sampled_ids = model.generate(
        **inputs,
        do_sample=True,
        temperature=0.8, # 1보다 낮은 0.8을 지정해서 확률이 높은 토큰에 조금 더 가중치를 부여
        top_k=40, # 확률이 가장 높은 토큰 40개만 후보로 설정
        top_p=0.9,
        **generation_options,
    )

    # 생성 결과에는 원래 입력 ID와 새 토큰 ID가 모두 들어 있으므로 전체를 문자열로 복원한다.
# skip_special_tokens=True는 EOS와 패딩 같은 제어용 토큰을 화면 문자열에서 제외한다.
greedy_text = tokenizer.decode(greedy_ids[0], skip_special_tokens=True)
sampled_text = tokenizer.decode(sampled_ids[0], skip_special_tokens=True)

print("[Greedy]")
print(greedy_text)
print()
print("[Sampling]")
print(sampled_text)

[Greedy]
Natural language processing helps computers learn to recognize and understand words.
"We're seeing a lot of new ways that people can use the Internet,"

[Sampling]
Natural language processing helps computers understand and use a vast array of languages. But it also keeps the human mind on track, so that even if we


## 07. 다음 단원과 연결하기

이번 실습의 공통 중심은 `문맥 → 어휘 로짓 → 다음 토큰 확률 → 토큰 선택`이다. 뒤의 모델들은 이 흐름을 버리는 것이 아니라 **문맥을 표현하는 방법**과 **예측에 사용할 정보의 범위**를 바꾼다.

- **N-gram**은 최근 `n-1`개 토큰의 등장 횟수로 조건부확률을 계산한다. 다음 노트북에서 smoothing과 perplexity까지 직접 확인한다.
- **NNLM**은 고정 길이 문맥의 토큰 ID를 임베딩과 신경망으로 처리하고, 정답 다음 토큰과의 손실로 가중치를 학습한다.
- **Seq2Seq**의 Decoder는 이전 정답 또는 생성 토큰뿐 아니라 Encoder가 만든 입력 문장 정보도 조건으로 사용한다. 학습 때 정답 이전 토큰을 넣는 Teacher Forcing과 추론 때 자신의 출력을 다시 넣는 차이가 중요하다.
- **GPT**는 앞 토큰만 보는 causal language modeling으로 다음 토큰 예측을 대규모 데이터에서 사전학습한다. 이번 GPT-2 실습이 이 경로의 추론 예시이다.
- **BERT**는 왼쪽과 오른쪽 문맥을 함께 보고 가린 토큰을 복원하는 masked language modeling을 사용한다. 문맥 표현과 분류에는 강하지만 GPT처럼 왼쪽부터 다음 토큰을 반복 선택하는 생성 방식과는 다르다.


### GPT-2 추론 흐름 한눈에 보기

아래 그림은 입력 ID `(1, 5)`가 전체 로짓 `(1, 5, 50257)`로 바뀐 뒤 마지막 위치를 선택하고, softmax와 top-k를 거쳐 다음 토큰을 정하는 흐름을 정리한다. 선택한 토큰을 문맥에 다시 붙이는 고리가 자동회귀 생성이며, 이 과정은 사전학습 GPT-2의 추론이므로 가중치를 새로 학습하지 않는다.

![GPT-2 다음 토큰 추론 흐름](attachment:gpt2_next_token_inference.png)

그림은 왼쪽 위에서 시작해 화살표를 따라 읽는다.

1. **입력 ID `(1, 5)`**
   - 문장 1개가 5개의 토큰 ID로 변환된 상태이다.
   - `B=1`은 배치의 문장 수, `T=5`는 입력 토큰 수이다.

2. **전체 로짓 `(1, 5, 50257)`**
   - GPT-2가 5개 토큰 위치마다 어휘 50,257개에 대한 다음 토큰 점수를 만든다.
   - `V=50257`은 GPT-2의 어휘 크기이다.

3. **마지막 위치 선택 `(1, 50257)`**
   - 현재 문장 바로 다음에 올 토큰을 예측하기 위해 마지막 입력 위치의 로짓만 선택한다.
   - 토큰 위치 축 `T`가 사라지고 어휘 후보 점수만 남는다.

4. **Softmax**
   - 50,257개의 로짓을 합이 1인 다음 토큰 확률로 변환한다.

5. **Top-k 후보 확인**
   - 확률이 높은 후보를 순서대로 확인한다.
   - 그림에서는 `' learn'`, `' understand'`, `' to'` 등이 주요 후보이다.

6. **토큰 선택**
   - **Greedy**는 확률이 가장 높은 top-1 토큰을 선택한다.
   - **Sampling**은 `temperature`, `top_k`, `top_p`로 후보 분포를 조절한 뒤 확률적으로 토큰을 선택한다.

7. **선택한 토큰을 문맥에 추가**
   - 선택한 토큰 ID를 기존 입력 뒤에 붙인다.

8. **과정 반복**
   - 길어진 문맥을 다시 GPT-2에 입력하여 다음 토큰을 계속 생성한다.

```text
토큰 ID 입력
→ 어휘 전체의 로짓
→ 마지막 위치 선택
→ 다음 토큰 확률
→ Greedy 또는 Sampling
→ 선택한 토큰을 문맥에 추가
→ 반복
```

`no training`은 현재 실습에서 GPT-2를 새로 학습하지 않고 사전학습된 가중치로 추론만 한다는 뜻이다.

확률이 높은 다음 토큰은 학습 데이터에서 현재 문맥 뒤에 나타날 가능성이 높다는 의미일 뿐, 생성 내용의 사실성을 보장하지 않는다.